# Final Statistical Comparison — Paired

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_classifier_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
from final_classifier_evaluation import *

DRY_RUN = True
RECOMPUTE_TEST_PREDICTIONS = False
ALLOW_UNVERIFIED_LEGACY_PREDICTIONS = False
TEST_BATCH_SIZE = 8
TEST_NUM_WORKERS = 4
DEVICE = "auto"
PATIENT_AGGREGATION = "mean"
TEST_CSV = PROJECT_ROOT / "data/processed/metadata/test.csv"
TEST_DATASET_MANIFEST = PROJECT_ROOT / "results/final_evaluation/test_dataset_manifest.json"
REGISTRY_PATH = PROJECT_ROOT / "configs/final_classifier_registry.json"

DRY_RUN = True
BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95
FINAL_RANKING_PRIMARY = "test_roc_auc"
FINAL_RANKING_SECONDARY = "test_pr_auc"
CALIBRATION_METRICS = ["Brier score", "ECE", "reliability curve"]
OUTPUT_DIR = PROJECT_ROOT / "results/final_evaluation"
PREDICTION_DIR = OUTPUT_DIR / "test_predictions"

In [ ]:
# Scientific-only read: must list blockers and missing predictions without crashing even
# while the lock is operationally incomplete.
manifest = validate_locked_finalists_manifest(OUTPUT_DIR / "finalists_manifest.json", require_operational_complete=False)
expected = [x["experiment_id"] for x in manifest["finalists"]]
available = [eid for eid in expected if (PREDICTION_DIR / f"{eid}.csv").is_file()]
missing = sorted(set(expected) - set(available))
print("Modelli paired:", available)
print("scientific_selection_complete:", manifest.get("scientific_selection_complete"), "| final_aggregation_complete:", manifest.get("final_aggregation_complete"))
print("Blocker operativi:", manifest.get("operational_blockers")); print("Mancanti:", missing)
if DRY_RUN: print("DRY_RUN: nessun test statistico o figura scritto.")

In [ ]:
if not DRY_RUN:
    # Operational gate: only proceed once every finalist has real, verified test predictions.
    manifest = validate_locked_finalists_manifest(OUTPUT_DIR / "finalists_manifest.json", require_operational_complete=True)
    if missing: raise RuntimeError(f"Predizioni centralizzate mancanti: {missing}")
    import itertools, matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
    canonical = pd.read_csv(TEST_CSV); canonical_ids = canonical.patient_id.astype(str)
    finalist_by_id = {x["experiment_id"]: x for x in manifest["finalists"]}
    frames = {}
    for eid in expected:
        source_manifest_path = PREDICTION_DIR / f"{eid}.manifest.json"
        if not source_manifest_path.is_file(): raise RuntimeError(f"Manifest centralizzato mancante: {source_manifest_path}")
        payload = json.loads(source_manifest_path.read_text())
        finalist, pred_path = finalist_by_id[eid], PREDICTION_DIR / f"{eid}.csv"
        required = {"experiment_id": eid, "finalists_lock_signature": manifest["lock_signature"], "test_dataset_manifest_signature": content_signature(TEST_DATASET_MANIFEST), "validation_threshold": finalist["validation_threshold"], "threshold_method": finalist["threshold_method"]}
        incompatible = [key for key, value in required.items() if payload.get(key) != strict_jsonable(value)]
        if payload.get("prediction_file_signature") != content_signature(pred_path): incompatible.append("prediction_file_signature")
        level = payload.get("provenance_level", "invalid")
        if level == "legacy_normalized_unverified" and not ALLOW_UNVERIFIED_LEGACY_PREDICTIONS: incompatible.append("provenance_level")
        if level not in {"verified_native", "verified_recomputed", "legacy_normalized_unverified"}: incompatible.append("provenance_level")
        if incompatible: raise RuntimeError(f"Manifest centralizzato incompatibile per {eid}: {sorted(set(incompatible))}")
        frames[eid] = pd.read_csv(pred_path)
    aligned = compare_patient_sets(frames, canonical_ids); y = next(iter(aligned.values())).y_true.to_numpy()
    rows = []
    registry = {x["experiment_id"]: x for x in build_experiment_registry(REGISTRY_PATH)}
    for a, b in itertools.combinations(expected, 2):
        ga, gb = registry[a].get("primary_comparison_group"), registry[b].get("primary_comparison_group")
        family = "primary" if ga and ga == gb else "secondary"
        sa, sb = aligned[a].y_score.to_numpy(), aligned[b].y_score.to_numpy()
        boot = paired_stratified_bootstrap(y, sa, sb, "roc_auc", BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED, CI_LEVEL); boot_pr = paired_stratified_bootstrap(y, sa, sb, "pr_auc", BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED, CI_LEVEL); dl = delong_roc_test(y, sa, sb)
        pa, pb = aligned[a].y_pred.to_numpy(), aligned[b].y_pred.to_numpy(); mc = mcnemar_test(y, pa, pb)
        # One row per (pair, metric): ROC-AUC keeps DeLong as its paired p-value, PR-AUC uses the
        # bootstrap p-value (DeLong has no PR-AUC equivalent), McNemar is its own metric/row
        # instead of two columns bolted onto every row -- never mix different statistical tests
        # into the same Holm correction family.
        rows.append({"model_a": a, "model_b": b, "metric": "roc_auc", "comparison_family": family, **boot, "delong_p_value": dl["p_value"], "raw_p_value": dl["p_value"]})
        rows.append({"model_a": a, "model_b": b, "metric": "pr_auc", "comparison_family": family, **boot_pr, "raw_p_value": boot_pr["p_bootstrap"]})
        rows.append({"model_a": a, "model_b": b, "metric": "mcnemar", "comparison_family": family, "mcnemar_b": mc["b"], "mcnemar_c": mc["c"], "mcnemar_statistic": mc["statistic"], "mcnemar_method": mc["method"], "raw_p_value": mc["p_value"]})
    comparisons = pd.DataFrame(rows)
    comparisons["holm_family"] = comparisons["comparison_family"] + "_" + comparisons["metric"]
    comparisons["holm_adjusted_p_value"] = float("nan")
    for _, group in comparisons.groupby("holm_family"):
        comparisons.loc[group.index, "holm_adjusted_p_value"] = holm_adjustment(group["raw_p_value"])
    comparisons["significant_raw"] = comparisons.raw_p_value < .05; comparisons["significant_holm"] = comparisons.holm_adjusted_p_value < .05
    comparisons.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False); (OUTPUT_DIR / "paired_comparisons.json").write_text(strict_json_dumps(comparisons.to_dict("records"), indent=2) + "\n")
    ranking = pd.DataFrame([{"experiment_id": eid, "test_roc_auc": roc_auc_score(y, aligned[eid].y_score), "test_pr_auc": average_precision_score(y, aligned[eid].y_score)} for eid in expected]).sort_values(["test_roc_auc", "test_pr_auc"], ascending=False); ranking.to_csv(OUTPUT_DIR / "final_ranking.csv", index=False)
    figures = OUTPUT_DIR / "figures"; figures.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 6));
    for eid in expected:
        fpr, tpr, _ = roc_curve(y, aligned[eid].y_score); ax.plot(fpr, tpr, label=f"{eid} ({roc_auc_score(y, aligned[eid].y_score):.3f})")
    ax.plot([0,1],[0,1], "k--"); ax.legend(fontsize=7); ax.set(xlabel="1 - specificity", ylabel="sensitivity"); fig.tight_layout(); fig.savefig(figures / "final_roc_curves.png", dpi=300); plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 6));
    for eid in expected:
        precision, recall, _ = precision_recall_curve(y, aligned[eid].y_score); ax.plot(recall, precision, label=f"{eid} ({average_precision_score(y, aligned[eid].y_score):.3f})")
    ax.axhline(y.mean(), ls="--", color="k"); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(figures / "final_pr_curves.png", dpi=300); plt.close(fig)
    from sklearn.calibration import calibration_curve
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, ax = plt.subplots(figsize=(8, 6))
    for eid in expected:
        frac, mean = calibration_curve(y, aligned[eid].y_score, n_bins=10, strategy="uniform"); ax.plot(mean, frac, marker="o", label=eid)
    ax.plot([0,1],[0,1], "k--"); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(figures / "calibration_curves.png", dpi=300); plt.close(fig)
    fig, axes = plt.subplots(len(expected), 1, figsize=(6, 4 * len(expected)), squeeze=False)
    for ax, eid in zip(axes[:,0], expected): ConfusionMatrixDisplay.from_predictions(y, aligned[eid].y_pred, ax=ax); ax.set_title(eid)
    fig.tight_layout(); fig.savefig(figures / "confusion_matrices.png", dpi=300); plt.close(fig)
    roc_boot = comparisons[comparisons.metric.eq("roc_auc")].copy(); fig, ax = plt.subplots(figsize=(9, max(4, len(roc_boot)*.35))); ypos = range(len(roc_boot)); ax.errorbar(roc_boot.mean_difference, ypos, xerr=[roc_boot.mean_difference-roc_boot.ci_lower, roc_boot.ci_upper-roc_boot.mean_difference], fmt="o"); ax.axvline(0, color="k", ls="--"); ax.set_yticks(list(ypos), [f"{a} - {b}" for a,b in zip(roc_boot.model_a, roc_boot.model_b)]); fig.tight_layout(); fig.savefig(figures / "bootstrap_auc_differences.png", dpi=300); plt.close(fig)
    best = ranking.iloc[0]; significant = comparisons[comparisons.significant_holm.fillna(False)]; nonsignificant = comparisons[~comparisons.significant_holm.fillna(False)]
    legacy = [eid for eid in expected if json.loads((PREDICTION_DIR / f"{eid}.manifest.json").read_text()).get("provenance_level") == "legacy_normalized_unverified"]
    resnet_absent = not any(registry[eid]["architecture"] == "ResNet-50" for eid in expected)
    conclusions = f"# Conclusioni finali\n\nMiglior valore puntuale: **{best.experiment_id}**, ROC-AUC {best.test_roc_auc:.4f} e PR-AUC {best.test_pr_auc:.4f}. Il ranking puntuale non implica superiorità statistica.\n\nConfronti paired significativi dopo Holm: {len(significant)}; non significativi: {len(nonsignificant)}. Intervalli bootstrap e p-value sono negli artefatti paired.\n\nProvenance legacy accettata: {legacy or 'nessuna'}. ResNet assente dal confronto finale: {resnet_absent}.\n\n## Limiti metodologici\n\nIl test era già stato osservato durante lo sviluppo. Nessuna vittoria è dichiarata senza supporto paired corretto per molteplicità; serve conferma esterna o grouped cross-validation.\n"
    (OUTPUT_DIR / "final_conclusions.md").write_text(conclusions)